# 00 — Synthetic DGP Exploration

Mirrors `04_synthetic_data_generation.ipynb` (original).
Explores the synthetic DGP structure, sub-group distributions, and the
heterogeneity sweep.  No outputs are saved — this is a reference notebook only.

Run this once to understand the DGP before running the experiment.

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
REVISED_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for p in [PROJECT_ROOT, REVISED_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from cdv_utils.synthetic_dgp import (
    generate_synthetic_dataset, get_w_cols,
    SUBGROUP_FEATURES, DEFAULT_VARIANT_SHARES,
    SG0_STRUCTURAL_SUBGROUPS, MISSING_VALUE
)
from cdv_utils.analysis_utils import create_synthetic_variant_elbow_chart

print('Libraries loaded.')

## 1. Generate Sample Dataset

In [ ]:
df = generate_synthetic_dataset(n=10_000, alpha=0.5, seed=42)
w_cols = get_w_cols()
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Feature columns: {w_cols}')

## 2. Sub-group Distribution

In [ ]:
sg_counts = df['subgroup'].value_counts().sort_index()
for sg in range(6):
    features = SUBGROUP_FEATURES[sg]
    print(f'SG{sg}: n={sg_counts.get(sg,0):4d} ({sg_counts.get(sg,0)/len(df)*100:5.1f}%) | {features}')

print(f'\nTop-3 coverage: {sg_counts.iloc[:3].sum()/len(df)*100:.1f}%')

## 3. Elbow Chart

In [ ]:
sorted_patterns, counts, cumulative_pct, total = create_synthetic_variant_elbow_chart(df, w_cols)
plt.show()

## 4. Heterogeneity Sweep

In [ ]:
ALPHA_VALUES = [0.0, 0.25, 0.5, 0.75, 1.0]
fig, axes = plt.subplots(1, len(ALPHA_VALUES), figsize=(18, 4), sharey=True)

for idx, alpha in enumerate(ALPHA_VALUES):
    df_a = generate_synthetic_dataset(n=5000, alpha=alpha, seed=42)
    data_to_plot = [df_a[df_a['subgroup'] == sg]['ite'].values for sg in range(6)]
    bp = axes[idx].boxplot(data_to_plot, labels=[f'SG{i}' for i in range(6)], patch_artist=True)
    axes[idx].axhline(y=5, color='black', linestyle='--', linewidth=1, alpha=0.5)
    axes[idx].set_title(f'α = {alpha}')
    axes[idx].set_xlabel('Sub-group')
    if idx == 0:
        axes[idx].set_ylabel('ITE')

plt.suptitle('True CATE Distribution by Sub-group Across Heterogeneity Levels', fontsize=12)
plt.tight_layout()
plt.show()

summary = pd.DataFrame(index=[f'SG{i}' for i in range(6)])
for alpha in ALPHA_VALUES:
    df_a = generate_synthetic_dataset(n=5000, alpha=alpha, seed=42)
    summary[f'α={alpha}'] = [df_a[df_a['subgroup'] == sg]['ite'].mean() for sg in range(6)]
print('ITE mean by sub-group:')
display(summary.round(2))